- 监督学习用 forward KL、强化学习用 reverse KL
    - 监督学习更像最小化 $D(P\|Q)$ (从数据分布 $P$ 采样来拟合模型 $Q$）
        - 类似“模仿/监督”（从 𝑃 取样，提升 𝑄 在这些样本上的对数似然）；
    - 而最大熵 RL 更像最小化 $D(Q\|P)$ (从策略 $Q$ 采样去靠近“真分布”/最优分布 $P$）；
        - “带熵正则的反馈学习/单步 bandit”（从 𝑄 取样，最大化这些样本在“真分布”下的对数概率）

```python
log_q = log_softmax(logits, dim=-1)        # 模型分布 Q
log_p = log_softmax(ref_logits, dim=-1)    # 参考分布 P（ref_model）

# kl_div(input, target)
kl = F.kl_div(log_q, log_p, reduction="none", log_target=True)
```

$$D_{\mathrm{KL}}(P\parallel Q)= \sum_{v} P(v)\,[\log P(v)-\log Q(v)]$$

- $P=\mathrm{softmax}(\text{ref\_logits}), Q=\mathrm{softmax}(\text{logits})$
    - https://docs.pytorch.org/docs/stable/generated/torch.nn.KLDivLoss.html
    - 这里是 forward kl，在惩罚“当前模型 Q 在参考模型 P 有概率质量的地方给得太低”的情况——典型的 mode-covering 倾向。
- 是词表空间求和，不取平均；

- kl 的单位：nats/token
- KL散度的计算涉及到对两个概率分布中每一个对应元素的考量。在大型语言模型的上下文中，这两个概率分布是模型（model）和参考模型（ref_model）在词表（vocabulary）上预测下一个词元（token）的概率分布。

In [1]:
import torch
import torch.nn.functional as F

In [13]:
p = torch.tensor([0.9, .1])
q1 = torch.tensor([0.8, .2])
q2 = torch.tensor([0.6, .4])
q3 = torch.tensor([0.3, .7])

In [14]:
F.kl_div(torch.log(q1), torch.log(p), reduction='sum', log_target=True)

tensor(0.0367)

In [15]:
(p * torch.log(p / q1)).sum()

tensor(0.0367)

In [16]:
F.kl_div(torch.log(q2), torch.log(p), reduction='sum', log_target=True)

tensor(0.2263)

In [17]:
F.kl_div(torch.log(q3), torch.log(p), reduction='sum', log_target=True)

tensor(0.7942)

### kl loss 水平

- https://wandb.ai/lingchang-ustc/search_async_rl/workspace?nw=nwuserlingchang
    - 0 - 0.2 之间

### kl 散度的含义（kl on llm）

$$
D_{\mathrm{KL}}(P\parallel Q)=H(P,Q)-H(P)
$$
- 熵：$H(P)=-\mathbb E_{t\sim P}[\log P(t)]$，真分布自己的不确定性
- 交叉熵：$H(P,Q)=-\mathbb E_{t\sim P}[\log Q(t)]$，用模型 $Q$ 看待真分布 $P$ 的平均惊讶
- $D_{\mathrm{KL}}(P\parallel Q)=\mathbb E_{t\sim P}\!\big[\log P(t)-\log Q(t)\big]$

$$
\mathrm{ppl}(P)=\exp(H(P)),\quad \mathrm{ppl}(P,Q)=\exp(H(P,Q))
$$
- 困惑度定义为熵的指数：刻画的是模型面临的选择（选择越多，越迷茫）
    - 词表的均匀采样，$H=\ln|V|$, $\mathrm{ppl}=e^{\ln|V|}=|V|$（选择最大）
    - 如果分布很尖（几乎确定），$H\to 0, \mathrm{ppl}\to 1$

$$
\frac{\mathrm{ppl}(P,Q)}{\mathrm{ppl}(P)}
=\exp\!\big(D_{\mathrm{KL}}(P\parallel Q)\big)
$$

In [18]:
p = torch.tensor([0.9, 0.1], dtype=torch.float64)
q = torch.tensor([0.8, 0.2], dtype=torch.float64)

H_P  = -(p * p.log()).sum()
H_PQ = -(p * q.log()).sum()
KL_PQ = (p * (p.log() - q.log())).sum()

print("exp(H(P,Q)-H(P)) =", torch.exp(H_PQ - H_P).item())
print("exp(KL(P||Q))    =", torch.exp(KL_PQ).item())

exp(H(P,Q)-H(P)) = 1.0373714004169436
exp(KL(P||Q))    = 1.0373714004169436
